# Production Screener — Multi-Factor Stock Screener

**Philosophy:** Fraud/distress removal IS the alpha. We don't pick stocks — we remove
the ones that will blow up, then let cheap+quality compound.

**Pipeline:** Hard Gates (8) → Regression Scoring → Tree Agreement (≥0.55) → Top 15 Equal-Weight

**Validated:** WF CAGR +31.5%, Sharpe 1.45, MaxDD -8.1% in backtest ($10B cap, momentum gate)

---

## 1. Data Import

In [1]:
import sys
from pathlib import Path

import joblib
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from pipeline.feature_library import add_piotroski_ext, add_normalised_ratios
from modeling.constants import (
    BENEISH_THRESHOLD, TREE_THRESHOLD, PIOTROSKI_MIN,
    VALUE_GATE_PCT, ALTMAN_Z_MIN, MOMENTUM_12M_MIN,
    MAX_MARKET_CAP_PROD,
)

MIN_MARKET_CAP = 50_000_000

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

In [2]:
DATA_PATH = ROOT / 'data' / 'historical_dataset_clean.parquet'
MODELS_DIR = ROOT / 'models'

df_raw = pd.read_parquet(DATA_PATH)
df = df_raw[df_raw['period_type'] == 'annual'].copy()

# Enrich: Piotroski extended signals + normalised ratios
df = add_piotroski_ext(df)
df = add_normalised_ratios(df)

# Use latest complete fiscal year
LATEST_YEAR = df[df['market'] == 'US']['fiscal_year'].value_counts().sort_index()
LATEST_YEAR = LATEST_YEAR[LATEST_YEAR >= 100].index.max()
print(f'Dataset: {len(df):,} rows | Using fiscal year: {LATEST_YEAR}')
print(f'US stocks in {LATEST_YEAR}: {len(df[(df["market"]=="US") & (df["fiscal_year"]==LATEST_YEAR)]):,}')

Dataset: 58,190 rows | Using fiscal year: 2025
US stocks in 2025: 4,133


## 2. Hard Gates — Fraud & Distress Removal

This is where the alpha lives. Remove fraud, distress, and uninvestable names BEFORE scoring.

In [3]:
# Start with latest-year US stocks
universe = df[(df['market'] == 'US') & (df['fiscal_year'] == LATEST_YEAR)].copy()
print(f'Starting universe: {len(universe)} US stocks')

# Gate 1: Beneish M-score (likely NOT a manipulator)
gate1 = universe[universe['beneish_m_score'] < BENEISH_THRESHOLD]
print(f'After Beneish < {BENEISH_THRESHOLD}:  {len(gate1)} ({len(universe)-len(gate1)} removed — fraud risk)')

# Gate 2: Piotroski F-score (minimum financial health)
gate2 = gate1[gate1['piotroski_f_score'] >= PIOTROSKI_MIN]
print(f'After Piotroski >= {PIOTROSKI_MIN}:   {len(gate2)} ({len(gate1)-len(gate2)} removed — weak fundamentals)')

# Gate 3: ROA positive (profitable operations)
gate3 = gate2[gate2['piotroski_roa_pos'] == 1]
print(f'After ROA positive:     {len(gate3)} ({len(gate2)-len(gate3)} removed — unprofitable)')

# Gate 4: Not a known fraud suspect
gate4 = gate3[gate3['fraud_suspect'] == 0]
print(f'After fraud_suspect=0:  {len(gate4)} ({len(gate3)-len(gate4)} removed — suspected fraud)')

# Gate 5: Market cap (investable, avoid mega-cap institutional territory)
gate5 = gate4[
    (gate4['market_cap_at_filing'] >= MIN_MARKET_CAP) &
    (gate4['market_cap_at_filing'] <= MAX_MARKET_CAP_PROD)
]
print(f'After mkt_cap ${MIN_MARKET_CAP/1e6:.0f}M-${MAX_MARKET_CAP_PROD/1e9:.0f}B: {len(gate5)} ({len(gate4)-len(gate5)} removed)')

# Gate 6: Altman Z-score (not in extreme distress)
gate6 = gate5[gate5['altman_z_score'] > ALTMAN_Z_MIN]
print(f'After Altman Z > {ALTMAN_Z_MIN}:   {len(gate6)} ({len(gate5)-len(gate6)} removed — distress zone)')

# Gate 7: Value — not grossly overpriced vs sector peers
gate7 = gate6[gate6['ps_ratio_sector_pct'].fillna(0.5) <= VALUE_GATE_PCT]
print(f'After P/S sector<={VALUE_GATE_PCT:.0%}: {len(gate7)} ({len(gate6)-len(gate7)} removed — overpriced)')

survivors = gate7.copy()
print(f'\n✓ {len(survivors)} survivors pass all hard gates ({len(universe)-len(survivors)} eliminated)')

Starting universe: 4133 US stocks
After Beneish < -1.78:  3625 (508 removed — fraud risk)
After Piotroski >= 3:   2545 (1080 removed — weak fundamentals)
After ROA positive:     1784 (761 removed — unprofitable)
After fraud_suspect=0:  1780 (4 removed — suspected fraud)
After mkt_cap $50M-$10B: 813 (967 removed)
After Altman Z > 1.0:   623 (190 removed — distress zone)
After P/S sector<=70%: 480 (143 removed — overpriced)

✓ 480 survivors pass all hard gates (3653 eliminated)


In [4]:
# Gate 8: Momentum — not in structural freefall
before_mom = len(survivors)
survivors = survivors[survivors['momentum_12m_prior'].fillna(0) > MOMENTUM_12M_MIN]
print(f'After momentum > {MOMENTUM_12M_MIN:.0%}:  {len(survivors)} ({before_mom - len(survivors)} removed — falling knives)')
print(f'\n✓ {len(survivors)} survivors pass all hard gates (incl. momentum)')

After momentum > -40%:  435 (45 removed — falling knives)

✓ 435 survivors pass all hard gates (incl. momentum)


## 3. Feature Selection — 22 Canonical Features (3y Horizon)

In [5]:
# Load canonical feature set from model metadata
with open(MODELS_DIR / 'model_meta.json') as f:
    meta = json.load(f)

FEATURES_3Y = meta['3y']['features']
print(f'3y LightGBM features ({len(FEATURES_3Y)}):')
for i, feat in enumerate(FEATURES_3Y, 1):
    print(f'  {i:2d}. {feat}')

3y LightGBM features (28):
   1. pb_ratio_sector_pct
   2. pb_ratio
   3. ps_ratio_sector_pct
   4. ps_ratio
   5. vol_rank_12m
   6. altman_x4
   7. book_to_market
   8. piotroski_shares_ok
   9. shares_at_filing
  10. filing_lag_days
  11. debt_growth_yoy
  12. days_inventory
  13. other_noncurrent_assets
  14. debt_to_assets_sector_pct
  15. beneish_m_score_sector_pct
  16. ap_growth
  17. gross_margin_sector_pct
  18. gross_margin
  19. beneish_lvgi
  20. noa_to_assets
  21. capex_growth
  22. piotroski_delta_liq
  23. intangibles_to_assets
  24. beneish_dsri
  25. days_payable
  26. assets_growth_sector_pct
  27. ppe_growth_yoy
  28. revenue_growth_sector_pct


In [6]:
# Check feature availability in survivors
missing = [f for f in FEATURES_3Y if f not in survivors.columns]
if missing:
    print(f'WARNING: Missing features: {missing}')
else:
    print(f'✓ All {len(FEATURES_3Y)} features available')

# Median-impute missing values (same as training)
train_medians = meta['3y']['train_medians']
for feat in FEATURES_3Y:
    if feat in survivors.columns:
        median_val = train_medians.get(feat, 0)
        survivors[feat] = survivors[feat].fillna(median_val)

null_pct = survivors[FEATURES_3Y].isnull().sum().sum() / (len(survivors) * len(FEATURES_3Y))
print(f'Post-imputation null rate: {null_pct:.1%}')

✓ All 28 features available
Post-imputation null rate: 0.0%


## 4. Model Scoring — LightGBM + Decision Tree

In [7]:
# Load production models
lgbm_3y = joblib.load(MODELS_DIR / 'model_3y.joblib')
reg_3y_path = MODELS_DIR / 'model_3y_regression.joblib'
tree_dict = joblib.load(MODELS_DIR / 'research_tree_snapshot.joblib')
tree_model = tree_dict['tree']
tree_features = tree_dict['features']

assert reg_3y_path.exists(), f'Missing regression model: {reg_3y_path}. Run modeling/train.py first.'

print(f'LightGBM 3y: {lgbm_3y.n_estimators_} trees, {len(FEATURES_3Y)} features')
print(f'Regression model: {reg_3y_path.name}')
print(f'Decision tree: depth {tree_model.get_depth()}, {len(tree_features)} features')
print(f'Agreement threshold: {TREE_THRESHOLD} (from modeling/constants.py)')
print(f'Ranking: LightGBM regression (return magnitude) — loaded from disk')

LightGBM 3y: 600 trees, 28 features
Regression model: model_3y_regression.joblib
Decision tree: depth 4, 28 features
Agreement threshold: 0.55 (from modeling/constants.py)
Ranking: LightGBM regression (return magnitude) — loaded from disk


In [8]:
# Score with LightGBM REGRESSION (predicted 3y return magnitude)
# Loads persisted model — trained once in modeling/train.py for reproducibility
reg_3y = joblib.load(MODELS_DIR / 'model_3y_regression.joblib')

# Impute using train medians from model metadata (not survivors.median())
reg_meta = meta.get('3y_regression', meta['3y'])
reg_medians = reg_meta['train_medians']
for feat in FEATURES_3Y:
    if feat in survivors.columns:
        survivors[feat] = survivors[feat].fillna(reg_medians.get(feat, 0))

# Score survivors with regression
X_surv = survivors[FEATURES_3Y].values
survivors['reg_3y'] = reg_3y.predict(X_surv)

# Score with decision tree (agreement signal)
for f in tree_features:
    if f not in survivors.columns:
        survivors[f] = 0
X_tree = survivors[tree_features].fillna(0).values
survivors['tree_prob'] = tree_model.predict_proba(X_tree)[:, 1]

print(f'Scored {len(survivors)} stocks (regression model loaded from disk)')
print(f'reg_3y range: [{survivors["reg_3y"].min():.3f}, {survivors["reg_3y"].max():.3f}]')
print(f'tree_prob range: [{survivors["tree_prob"].min():.3f}, {survivors["tree_prob"].max():.3f}]')

Scored 435 stocks (regression model loaded from disk)
reg_3y range: [-0.033, 2.339]
tree_prob range: [0.294, 0.751]


/Users/mhoque/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/mhoque/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


## 5. Agreement Gate — Tree Probability >= 0.55

In [9]:
agreed = survivors[survivors['tree_prob'] >= TREE_THRESHOLD].copy()
print(f'Agreement gate (tree_prob >= {TREE_THRESHOLD}): {len(agreed)} stocks pass')
print(f'  Removed: {len(survivors) - len(agreed)} stocks where tree disagrees')

Agreement gate (tree_prob >= 0.55): 39 stocks pass
  Removed: 396 stocks where tree disagrees


## 6. Portfolio Construction — Top 15 Equal-Weight

In [10]:
TOP_N = 15
AUM = 200_000  # $200K retail AUM
MAX_PCT_ADTV = 0.01  # Max 1% of daily volume per position

# ADTV liquidity filter using actual trading volume data
monthly_px = pd.read_parquet(ROOT / 'data' / 'monthly_prices.parquet')
obs_end = pd.Timestamp(f'{LATEST_YEAR}-12-31')
obs_start = pd.Timestamp(f'{LATEST_YEAR}-09-30')
adtv_data = monthly_px[
    (monthly_px['date'] >= obs_start) & (monthly_px['date'] <= obs_end)
].groupby('ticker')['adtv_30d'].median().reset_index()
adtv_data.columns = ['ticker', 'adtv_est']

agreed_liq = agreed.merge(adtv_data, on='ticker', how='left')
position_size = AUM / TOP_N  # $13,333 per position

# Filter: position must be < 1% of daily volume
agreed_liq = agreed_liq[
    agreed_liq['adtv_est'].isna() | (agreed_liq['adtv_est'] * MAX_PCT_ADTV >= position_size)
]
print(f'After ADTV liquidity filter: {len(agreed_liq)} stocks (removed {len(agreed)-len(agreed_liq)} illiquid)')

# Rank by REGRESSION score (predicted 3y return magnitude)
portfolio = agreed_liq.nlargest(TOP_N, 'reg_3y').copy()

# Equal-weight allocation
portfolio['weight'] = 1.0 / min(TOP_N, len(portfolio))
portfolio['position_size'] = AUM * portfolio['weight']

print(f'\nPortfolio: Top {len(portfolio)} by predicted 3y return')
print(f'Equal weight: {portfolio["weight"].iloc[0]:.1%} per position (${portfolio["position_size"].iloc[0]:,.0f})')

After ADTV liquidity filter: 36 stocks (removed 3 illiquid)

Portfolio: Top 15 by predicted 3y return
Equal weight: 6.7% per position ($13,333)


## 7. Stock Analysis — Key Metrics Per Pick

In [11]:
DISPLAY_COLS = [
    'ticker', 'name', 'sector', 'ml_3y', 'tree_prob',
    'piotroski_f_score', 'altman_z_score', 'beneish_m_score',
    'market_cap_at_filing', 'book_to_market', 'ocf_to_assets',
    'financing_cashflow_to_assets', 'ps_ratio_sector_pct',
]

analysis = portfolio[DISPLAY_COLS].copy()
analysis['market_cap_M'] = analysis['market_cap_at_filing'] / 1e6
analysis = analysis.drop(columns=['market_cap_at_filing'])

print('='*80)
print('PORTFOLIO PICKS — Detailed Analysis')
print('='*80)
for i, row in analysis.iterrows():
    print(f"\n{'─'*60}")
    print(f"  {row['ticker']} — {row['name']}")
    print(f"  Sector: {row['sector']} | Market Cap: ${row['market_cap_M']:.0f}M")
    print(f"  ML Score: {row['ml_3y']:.3f} | Tree: {row['tree_prob']:.3f}")
    print(f"  Piotroski: {row['piotroski_f_score']:.0f} | Altman Z: {row['altman_z_score']:.2f} | Beneish: {row['beneish_m_score']:.2f}")
    print(f"  Book/Market: {row['book_to_market']:.2f} | OCF/Assets: {row['ocf_to_assets']:.3f}")
    print(f"  Thesis: Cheap (P/S pct={row['ps_ratio_sector_pct']:.0%}), profitable (OCF+), not fraud")

PORTFOLIO PICKS — Detailed Analysis

────────────────────────────────────────────────────────────
  CHGG — CHEGG, INC
  Sector: Health & Professional Services | Market Cap: $65M
  ML Score: 0.624 | Tree: 0.598
  Piotroski: 5 | Altman Z: 12.44 | Beneish: -3.63
  Book/Market: 13.87 | OCF/Assets: 0.283
  Thesis: Cheap (P/S pct=4%), profitable (OCF+), not fraud

────────────────────────────────────────────────────────────
  ADNT — Adient plc
  Sector: Manufacturing | Market Cap: $1496M
  ML Score: 0.705 | Tree: 0.579
  Piotroski: 7 | Altman Z: 1.81 | Beneish: -2.45
  Book/Market: 1.43 | OCF/Assets: 0.071
  Thesis: Cheap (P/S pct=6%), profitable (OCF+), not fraud

────────────────────────────────────────────────────────────
  WKC — WORLD KINECT CORP
  Sector: Wholesale Trade | Market Cap: $1414M
  ML Score: 0.784 | Tree: 0.751
  Piotroski: 6 | Altman Z: 7.45 | Beneish: -2.43
  Book/Market: 1.38 | OCF/Assets: 0.040
  Thesis: Cheap (P/S pct=9%), profitable (OCF+), not fraud

─────────────────

## 8. LLM Summary — Buy Rationale Per Stock

Template-based rationale (no API call required). Each pick gets a structured thesis.

In [12]:
def generate_rationale(row):
    """Generate buy rationale from quantitative signals."""
    signals = []

    # Valuation
    if row.get('book_to_market', 0) > 0.5:
        signals.append('deep value (B/M > 0.5)')
    elif row.get('book_to_market', 0) > 0.3:
        signals.append('moderate value')
    if row.get('ps_ratio_sector_pct', 1) < 0.3:
        signals.append(f'cheap vs sector (P/S bottom {row["ps_ratio_sector_pct"]:.0%})')

    # Quality
    pio = row.get('piotroski_f_score', 0)
    if pio >= 7:
        signals.append(f'excellent quality (Piotroski {pio:.0f}/9)')
    elif pio >= 5:
        signals.append(f'solid quality (Piotroski {pio:.0f}/9)')

    # Cash flow
    ocf = row.get('ocf_to_assets', 0)
    if ocf > 0.10:
        signals.append(f'strong cash generation ({ocf:.1%} of assets)')
    elif ocf > 0.05:
        signals.append('positive operating cash flow')

    # Safety
    az = row.get('altman_z_score', 0)
    if az > 3.0:
        signals.append(f'very safe (Altman Z={az:.1f})')
    elif az > 2.0:
        signals.append(f'financially stable (Altman Z={az:.1f})')

    # Size
    mcap = row.get('market_cap_at_filing', 0) / 1e6
    if mcap < 500:
        signals.append(f'small-cap ${mcap:.0f}M — under institutional radar')
    elif mcap < 2000:
        signals.append(f'mid-small ${mcap:.0f}M')

    # ML confidence
    ml = row.get('ml_3y', 0)
    if ml > 0.6:
        signals.append(f'high ML confidence ({ml:.0%} beat prob)')
    elif ml > 0.5:
        signals.append(f'ML favourable ({ml:.0%} beat prob)')

    return '; '.join(signals) if signals else 'Passes all quantitative gates'


print('='*80)
print('BUY RATIONALE — Per-Stock Thesis')
print('='*80)
for _, row in portfolio.iterrows():
    rationale = generate_rationale(row)
    print(f"\n{row['ticker']} ({row['name']})")
    print(f"  → {rationale}")

BUY RATIONALE — Per-Stock Thesis

CHGG (CHEGG, INC)
  → deep value (B/M > 0.5); cheap vs sector (P/S bottom 4%); solid quality (Piotroski 5/9); strong cash generation (28.3% of assets); very safe (Altman Z=12.4); small-cap $65M — under institutional radar; high ML confidence (62% beat prob)

ADNT (Adient plc)
  → deep value (B/M > 0.5); cheap vs sector (P/S bottom 6%); excellent quality (Piotroski 7/9); positive operating cash flow; mid-small $1496M; high ML confidence (71% beat prob)

WKC (WORLD KINECT CORP)
  → deep value (B/M > 0.5); cheap vs sector (P/S bottom 9%); solid quality (Piotroski 6/9); very safe (Altman Z=7.5); mid-small $1414M; high ML confidence (78% beat prob)

ALTG (ALTA EQUIPMENT GROUP INC.)
  → deep value (B/M > 0.5); cheap vs sector (P/S bottom 11%); solid quality (Piotroski 5/9); small-cap $213M — under institutional radar; high ML confidence (73% beat prob)

SNEX (StoneX Group Inc.)
  → moderate value; cheap vs sector (P/S bottom 3%); financially stable (Altman Z

## 9. Final Output — Portfolio Summary Table

In [13]:
# Build final output table
output = portfolio[['ticker', 'name', 'sector', 'reg_3y', 'tree_prob',
                    'piotroski_f_score', 'altman_z_score', 'beneish_m_score',
                    'market_cap_at_filing']].copy()

output = output.rename(columns={
    'piotroski_f_score': 'piotroski',
    'altman_z_score': 'altman_z',
    'beneish_m_score': 'beneish_m',
    'market_cap_at_filing': 'market_cap',
})

output['market_cap'] = output['market_cap'] / 1e6  # Convert to $M

# Sort by reg_3y descending
output = output.sort_values('reg_3y', ascending=False).reset_index(drop=True)
output.index = output.index + 1  # 1-indexed rank

print('='*80)
print(f'PRODUCTION PORTFOLIO — Top {len(output)} Picks (FY{LATEST_YEAR})')
print(f'Strategy: Hard gates (8) → Regression ranking → Tree agreement (>= {TREE_THRESHOLD})')
print(f'Config: Beneish<-1.78, Pio>=3, ROA+, $50M-$10B, AltZ>1, P/S<=70th, Mom>-40%, tree>={TREE_THRESHOLD}')
print(f'WF backtest: CAGR +31.5%, Sharpe 1.45, MaxDD -8.1% (2013-2023)')
print('='*80)
print()

fmt = output.copy()
fmt['market_cap'] = fmt['market_cap'].apply(lambda x: f'${x:,.0f}M')
print(fmt.to_string())

PRODUCTION PORTFOLIO — Top 15 Picks (FY2025)
Strategy: Hard gates (8) → Regression ranking → Tree agreement (>= 0.55)
Config: Beneish<-1.78, Pio>=3, ROA+, $50M-$10B, AltZ>1, P/S<=70th, Mom>-40%, tree>=0.55
WF backtest: CAGR +31.5%, Sharpe 1.45, MaxDD -8.1% (2013-2023)

   ticker                           name                          sector  reg_3y  tree_prob  piotroski  altman_z  beneish_m market_cap
1    CHGG                     CHEGG, INC  Health & Professional Services   2.339      0.598      5.000    12.442     -3.634       $65M
2    ADNT                     Adient plc                   Manufacturing   1.654      0.579      7.000     1.809     -2.450    $1,496M
3     WKC              WORLD KINECT CORP                 Wholesale Trade   1.368      0.751      6.000     7.455     -2.425    $1,414M
4    ALTG      ALTA EQUIPMENT GROUP INC.                 Wholesale Trade   1.339      0.579      5.000     1.549     -2.382      $213M
5    SNEX              StoneX Group Inc.               

In [14]:
# Portfolio-level statistics
print('\n' + '='*80)
print('PORTFOLIO STATISTICS')
print('='*80)
print(f'  Positions:          {len(output)}')
print(f'  Avg reg score:      {output["reg_3y"].mean():.3f}')
print(f'  Avg tree prob:      {output["tree_prob"].mean():.3f}')
print(f'  Avg Piotroski:      {output["piotroski"].mean():.1f}')
print(f'  Avg Altman Z:       {output["altman_z"].mean():.2f}')
print(f'  Median market cap:  ${output["market_cap"].median():,.0f}M')
print(f'\n  Sectors:')
for sector, count in output['sector'].value_counts().items():
    print(f'    {sector}: {count}')

print(f'\n  Walk-forward backtest (Regression + momentum gate, tree>=0.55, $10B cap):')
print(f'    • Train: expanding window, clean data only')
print(f'    • Period: 2013-2023 (11 years)')
print(f'    • CAGR:  +31.5% vs SPY +13.6% (excess +17.9%)')
print(f'    • Sharpe: 1.45 | MaxDD: -8.1% | Beta: -0.18')
print(f'    • Survives after 150bps costs')
print(f'\n  Investment philosophy:')
print(f'    • Fraud/distress removal is the PRIMARY alpha')
print(f'    • Small/mid-cap ($50M-$10B) — under institutional radar')
print(f'    • Regression ranks by return MAGNITUDE (not just beat probability)')
print(f'    • Tree gate at 0.55 = strict quality control on what regression picks')
print(f'    • Momentum > -40% = no falling knives / structural decliners')
print(f'    • Equal-weight, annual rebalance')


PORTFOLIO STATISTICS
  Positions:          15
  Avg reg score:      1.022
  Avg tree prob:      0.616
  Avg Piotroski:      5.1
  Avg Altman Z:       3.26
  Median market cap:  $682M

  Sectors:
    Wholesale Trade: 4
    Manufacturing: 3
    Retail Trade: 3
    Finance: 2
    Health & Professional Services: 1
    Agriculture: 1
    Services: 1

  Walk-forward backtest (Regression + momentum gate, tree>=0.55, $10B cap):
    • Train: expanding window, clean data only
    • Period: 2013-2023 (11 years)
    • CAGR:  +31.5% vs SPY +13.6% (excess +17.9%)
    • Sharpe: 1.45 | MaxDD: -8.1% | Beta: -0.18
    • Survives after 150bps costs

  Investment philosophy:
    • Fraud/distress removal is the PRIMARY alpha
    • Small/mid-cap ($50M-$10B) — under institutional radar
    • Regression ranks by return MAGNITUDE (not just beat probability)
    • Tree gate at 0.55 = strict quality control on what regression picks
    • Momentum > -40% = no falling knives / structural decliners
    • Equal-wei

## 9.5 M&A Screen — LLM Flag

Uses Groq (free Llama 3.3 70B) to check if any picks have a pending acquisition/merger.
Requires: `pip install groq` + env var `GROQ_API_KEY` from console.groq.com.
Gracefully skips if unavailable.

In [15]:
import os

mna_flagged = []

try:
    from groq import Groq

    api_key = os.environ.get('GROQ_API_KEY')
    if not api_key:
        raise EnvironmentError('GROQ_API_KEY not set')

    client = Groq(api_key=api_key)

    # Build ticker list for the prompt
    ticker_list = [
        {'ticker': row['ticker'], 'name': row['name']}
        for _, row in output.iterrows()
    ]

    prompt = f"""You are a financial analyst. For each stock below, determine if there is
a currently PENDING acquisition, merger, going-private deal, or buyout offer as of today.

Stocks to check:
{json.dumps(ticker_list, indent=2)}

Respond with ONLY a JSON array (no markdown, no explanation):
[{{"ticker": "X", "pending_deal": true/false, "detail": "one-line summary or null"}}]

Rules:
- ONLY flag deals that are ANNOUNCED and PENDING (not rumored, not completed).
- If no pending deal, set pending_deal=false and detail=null.
- Be conservative: when uncertain, set pending_deal=false.
"""

    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
        max_tokens=2000,
    )
    raw = response.choices[0].message.content.strip()
    # Strip markdown code fences if present
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1].rsplit('```', 1)[0].strip()

    mna_results = json.loads(raw)
    mna_flagged = [r['ticker'] for r in mna_results if r.get('pending_deal')]

    print('M&A Screen (Llama 3.3 70B via Groq):')
    if mna_flagged:
        print(f'  PENDING DEALS DETECTED:')
        for r in mna_results:
            if r.get('pending_deal'):
                print(f"    {r['ticker']}: {r.get('detail', 'pending deal')}")
        print(f'\n  Action: Review flagged tickers — upside may be capped at deal price.')
    else:
        print('  No pending M&A deals detected in portfolio.')

except ImportError:
    print('M&A Screen: SKIPPED (install groq: pip install groq)')
except EnvironmentError as e:
    print(f'M&A Screen: SKIPPED ({e})')
except Exception as e:
    print(f'M&A Screen: SKIPPED (error: {e})')

# Store for downstream use
output['mna_flag'] = output['ticker'].isin(mna_flagged)

M&A Screen: SKIPPED (GROQ_API_KEY not set)


## 10. Persist Picks — JSON Tracking

Save picks to timestamped JSON (never overwritten) + latest.json for diffing across runs.

In [16]:
from datetime import date

today = date.today().isoformat()
picks_dir = ROOT / 'data'

# Build picks payload
picks_list = []
for _, row in output.iterrows():
    picks_list.append({
        'ticker': row['ticker'],
        'name': row['name'],
        'sector': row['sector'],
        'reg_3y': round(float(row['reg_3y']), 4),
        'tree_prob': round(float(row['tree_prob']), 4),
        'piotroski': int(row['piotroski']),
        'altman_z': round(float(row['altman_z']), 2),
        'market_cap': round(float(row['market_cap']), 1),
        'mna_flag': bool(row.get('mna_flag', False)),
    })

payload = {
    'date': today,
    'fiscal_year': int(LATEST_YEAR),
    'config': {
        'beneish_threshold': BENEISH_THRESHOLD,
        'piotroski_min': PIOTROSKI_MIN,
        'roa_positive': True,
        'market_cap_min': MIN_MARKET_CAP,
        'market_cap_max': MAX_MARKET_CAP_PROD,
        'altman_z_min': ALTMAN_Z_MIN,
        'ps_sector_pct_max': VALUE_GATE_PCT,
        'momentum_12m_min': MOMENTUM_12M_MIN,
        'tree_threshold': TREE_THRESHOLD,
        'top_n': TOP_N,
    },
    'picks': picks_list,
    'metadata': {
        'n_universe': int(len(universe)),
        'n_survivors': int(len(survivors)),
        'n_agreed': int(len(agreed)),
        'n_final': int(len(portfolio)),
        'mna_flagged': mna_flagged,
    },
}

# Save timestamped (never overwritten)
ts_path = picks_dir / f'production_picks_{today}.json'
if not ts_path.exists():
    ts_path.write_text(json.dumps(payload, indent=2))
    print(f'✓ Saved: {ts_path.name}')
else:
    print(f'⚠ Already exists (not overwritten): {ts_path.name}')

# Save latest (always overwritten)
latest_path = picks_dir / 'production_picks_latest.json'
prev_tickers = set()
if latest_path.exists():
    prev = json.loads(latest_path.read_text())
    prev_tickers = {p['ticker'] for p in prev.get('picks', [])}

latest_path.write_text(json.dumps(payload, indent=2))
print(f'✓ Saved: {latest_path.name}')

# Diff vs previous run
curr_tickers = {p['ticker'] for p in picks_list}
if prev_tickers:
    new_picks = curr_tickers - prev_tickers
    dropped = prev_tickers - curr_tickers
    print(f'\nDiff vs last run ({prev.get("date", "unknown")}):')
    print(f'  New:     {len(new_picks)} — {sorted(new_picks) if new_picks else "none"}')
    print(f'  Dropped: {len(dropped)} — {sorted(dropped) if dropped else "none"}')
    print(f'  Kept:    {len(curr_tickers & prev_tickers)}')
else:
    print('\nFirst run — no previous picks to diff against.')

✓ Saved: production_picks_2026-06-29.json


✓ Saved: production_picks_latest.json

Diff vs last run (2026-06-28):
  New:     0 — none
  Dropped: 0 — none
  Kept:    15
